# Precision Machining Quality Assurance Analysis

KAMP 정밀가공 품질보증 AI 데이터셋을 활용해 CNC 정밀가공 공정의 양품/불량을 분류하는 DNN 모델을 재현하고 정리한 개인 학습 프로젝트입니다.

This notebook follows the KAMP practice guide flow and keeps the analysis reproducible for a GitHub project.

## Data Source

중소벤처기업부, Korea AI Manufacturing Platform(KAMP), 정밀가공 품질보증 AI 데이터셋, 스마트제조혁신추진단(㈜인터엑스), 2022.12.23., www.kamp-ai.kr

## Before Running

The original CSV file is not included in this repository. Download it from KAMP and place it under `data/` before running the notebook.


## 1. 라이브러리 임포트

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from sklearn.preprocessing import Normalizer, MinMaxScaler
import tensorflow as tf
from tensorflow.keras.layers import Input, Dropout, Dense
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras import layers, losses
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


## 2. 데이터 로드 및 탐색

In [ ]:
# Load dataset
DATA_DIR = Path('data')
DATA_PATH_CANDIDATES = [
    DATA_DIR / path.name for path in DATA_DIR.glob('*.csv')
]
DATA_PATH_CANDIDATES.append(Path('precision_machining_quality_assurance.csv'))

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'CSV file was not found. Download the KAMP dataset and place the CSV file under data/.'
    )

data = pd.read_csv(DATA_PATH)
df = data.copy()
print(f'Loaded dataset: {DATA_PATH}')


In [ ]:
# 데이터 상위 5개 출력
df.head()

In [ ]:
# 양품/불량 개수 확인
df['passorfail'].value_counts()

sns.countplot(x = df['passorfail'], palette = 'coolwarm')
plt.title('Pass Or Fail Value Count')

In [ ]:
# 결측 개수 확인
df.isna().sum()

In [ ]:
# 데이터프레임 크기 확인
df.shape

In [ ]:
# 전처리 전 기초통계
df.describe().T

## 3. 변수 분포 시각화

In [ ]:
# 변수별 시각화
cols = df.columns.tolist()
plt.figure(figsize=(30, 30))
for i in range(len(cols)):
    if (cols[i] != 'SerialNo') & (cols[i] != 'ReceivedDateTime') & (cols[i] != 'passorfail'):
        plt.subplot(8, 5, i - 1)
        plt.title(cols[i])
        plt.hist(df[cols[i]])
plt.tight_layout()

In [ ]:
# coolwarm 컬러맵에서 색상 추출 (0.2는 푸른색 계열, 0.8은 붉은색 계열)
cmap = plt.cm.get_cmap('coolwarm')
color_ok = cmap(0.2)  # OK (파란색)
color_ng = cmap(0.8)  # NG (빨간색)

# 양품/불량별 시각화
cols = df.columns.tolist()
plt.figure(figsize=(30, 30))
pos_df = df[df['passorfail'] == 0]
neg_df = df[df['passorfail'] == 1]

for i in range(len(cols)):
    if (cols[i] != 'SerialNo') & (cols[i] != 'ReceivedDateTime') & (cols[i] != 'passorfail'):
        plt.subplot(8, 5, i - 1)
        plt.title(cols[i])

        # color 지정 및 투명도(alpha) 설정으로 겹치는 구간 시각화 개선
        plt.hist(pos_df[cols[i]], color=color_ok, alpha=0.7, label='OK')
        plt.hist(neg_df[cols[i]], color=color_ng, alpha=0.7, label='NG')

        plt.legend()
        plt.tight_layout()

## 4. 전처리

고정값 변수와 비수치형 변수를 제거하고, 이상치를 정리한다.

In [ ]:
# 고정 변수 제거
df = df[[col for col in df.columns if df[col].nunique() != 1]]

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(exclude='object').columns.tolist()
df = df[numeric_cols].copy()
df.head()


In [ ]:
# Remove outliers
# The original function returned inside the loop, so only the first numeric feature was processed.
# This version applies the same upper/lower quantile rule to all numeric feature columns.
def remove_outliers_by_quantile(data, target_col='passorfail', lower_q=0.001, upper_q=0.999):
    cleaned = data.copy()
    feature_cols = [
        col for col in cleaned.select_dtypes(include=np.number).columns
        if col != target_col
    ]

    for col in feature_cols:
        lower = cleaned[col].quantile(lower_q)
        upper = cleaned[col].quantile(upper_q)
        cleaned = cleaned[(cleaned[col] >= lower) & (cleaned[col] <= upper)]

    return cleaned.reset_index(drop=True)

out_df = remove_outliers_by_quantile(df)
print('Before:', df.shape)
print('After :', out_df.shape)
out_df.head()


## 5. Feature Selection (T-test)

To reduce test-set leakage, split the dataset first and run T-test feature selection only on the training data. Features with p-value below 0.1 are selected.


In [ ]:
# Split first, then run T-test only on training data
train_df, test_df = train_test_split(
    out_df,
    test_size=0.3,
    stratify=out_df['passorfail'],
    random_state=42,
)

def select_features_by_ttest(data, target_col='passorfail', alpha=0.1):
    selected_features = []
    for col in data.columns:
        if col == target_col:
            continue
        fail_group = data.loc[data[target_col] == 1, col]
        pass_group = data.loc[data[target_col] == 0, col]
        test_result = scipy.stats.ttest_ind(
            fail_group,
            pass_group,
            equal_var=False,
            nan_policy='omit',
        )
        if test_result.pvalue < alpha:
            selected_features.append(col)
    return selected_features

selected_features = select_features_by_ttest(train_df)
print('Selected features:', selected_features)

df_ttest = train_df[selected_features + ['passorfail']].copy()
df_ttest.head()


In [ ]:
import matplotlib.pyplot as plt

# 1. coolwarm 컬러맵에서 색상 추출 (최신 방식 적용)
cmap = plt.get_cmap('coolwarm')
color_ok = cmap(0.1)  # OK/Pass (진한 파란색 톤)
color_ng = cmap(0.9)  # NG/Fail (진한 빨간색 톤)

# 시각화할 컬럼 리스트 (타겟 변수는 제외해야 합니다)
# cols = df_ttest.columns.tolist()
# 예시: 만약 정답 컬럼 이름이 'Target'이라면 아래처럼 제외합니다.
cols = [col for col in df_ttest.columns if col != 'passorfail']

plt.figure(figsize=(20, 20))

for i, col in enumerate(cols):
    plt.subplot(4, 3, i + 1)
    plt.title(col, fontsize=14, fontweight='bold')

    # 2. OK(Pass)와 NG(Fail) 데이터 분리
    # ★ 주의: 'Target', 'Pass', 'Fail' 부분은 실제 데이터셋의 컬럼명과 값에 맞게 수정해 주세요!
    data_ok = df_ttest[df_ttest['passorfail'] == 0][col]
    data_ng = df_ttest[df_ttest['passorfail'] == 1][col]

    # 3. 각각의 히스토그램 그리기 (투명도 alpha와 테두리 edgecolor 적용)
    plt.hist(data_ok, color=color_ok, alpha=0.6, label='Pass (OK)', edgecolor='white')
    plt.hist(data_ng, color=color_ng, alpha=0.6, label='Fail (NG)', edgecolor='white')

    # 범례 및 배경 그리드 추가
    plt.legend(loc='upper right')
    plt.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
# plt.savefig('./Model/Histogram_vars.png', dpi=300, bbox_inches='tight') # 필요시 저장
plt.show()

In [ ]:
# 상관관계 시각화
plt.figure(figsize=(20, 15))
sns.heatmap(data=df_ttest.corr(), annot=True, fmt='.2f', linewidths=0.5, cmap='coolwarm')

## 6. Train/Test Split And SMOTE

Use the train/test split created before feature selection. SMOTE is applied only to the training data, and normalization/scaling parameters are fitted only on the training data.


In [ ]:
# Prepare train/test arrays
X_train = train_df[selected_features].copy()
X_test = test_df[selected_features].copy()
y_train = train_df['passorfail'].copy()
y_test = test_df['passorfail'].copy()

print('X_train', X_train.shape)
print('X_test', X_test.shape)
print('y_train', y_train.shape)
print('y_test', y_test.shape)


In [ ]:
# 학습 데이터 양품/불량 개수
print(y_train.value_counts())

sns.countplot(x = y_train, palette = 'coolwarm')
plt.title('y_train Value Count')

In [ ]:
# SMOTE 기반 학습 데이터 증강
smote = SMOTE(random_state=22, sampling_strategy={0: 2000, 1: 300}, k_neighbors=10)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(y_train_resampled.value_counts())

sns.countplot(x = y_train_resampled, palette = 'coolwarm')
plt.title('y_train_resampled Value Count')

In [ ]:
# 정규화 적용
normalizer = Normalizer()
scaler = MinMaxScaler()

X_train_norm = normalizer.fit_transform(X_train_resampled)
X_test_norm = normalizer.transform(X_test)
X_train_scaled = scaler.fit_transform(X_train_norm)
X_test_scaled = scaler.transform(X_test_norm)

## 7. 모델 정의 및 학습

심층 신경망을 정의하고 Stratified K-Fold 교차검증으로 학습한다.

In [ ]:
# 심층 신경망
def DNN_1(node_num, dropout_ratio):
    model = Sequential()
    model.add(Dense(node_num * 2, activation='relu'))
    model.add(Dense(node_num * 2, activation='relu'))
    model.add(Dropout(dropout_ratio))
    model.add(Dense(node_num, activation='relu'))
    model.add(Dense(node_num, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    return model

In [ ]:
Path('Model').mkdir(exist_ok=True)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
BATCH_SIZE = 32
n_iter = 1
acc_list = []
f1_list = []
for train_idx, valid_idx in skf.split(X_train_scaled, y_train_resampled):
    X_train, X_valid = X_train_scaled[train_idx], X_train_scaled[valid_idx]
    y_train, y_valid = y_train_resampled[train_idx], y_train_resampled[valid_idx]
    model = DNN_1(5, dropout_ratio=0.3)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.7, patience=3, verbose=1)
    es = EarlyStopping(monitor='val_loss', min_delta=0.00001, patience=20, verbose=1, mode='min', restore_best_weights=True)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    print('{} Fold'.format(n_iter))
    history = model.fit(X_train, y_train, callbacks=[es, reduce_lr], epochs=100, validation_data=(X_valid, y_valid))
    plt.plot(history.history['loss'], label='train loss')
    plt.plot(history.history['val_loss'], label='valid loss')
    plt.legend()
    plt.xlabel('Epoch'); plt.ylabel('loss')
    plt.show()
    model.save('./Model/{}th_fold.h5'.format(n_iter))
    test_predictions = model.predict(X_test_scaled, batch_size=BATCH_SIZE)
    f1 = f1_score(y_test, test_predictions > 0.5)
    f1_list.append(f1)
    n_iter += 1

In [ ]:
f1_list

## 8. 모델 평가

최적 모델을 불러와 혼동행렬로 성능을 확인한다.

In [ ]:
# 최적 모델 호출 및 혼동행렬 확인
best_model = tf.keras.models.load_model('./Model/1th_fold.h5')
test_predictions = best_model.predict(X_test_scaled, batch_size=BATCH_SIZE)
conf_matrix = confusion_matrix(y_test, test_predictions > 0.5)
plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix, xticklabels=[0, 1], yticklabels=[0, 1], annot=True, fmt='d', cmap = 'coolwarm')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Class'); plt.ylabel('True Class')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# 1. FPR, TPR, 임계값(Thresholds) 계산
# test_predictions는 확률값(0~1 사이)이어야 합니다.
fpr, tpr, thresholds = roc_curve(y_test, test_predictions)

# 2. AUC (Area Under Curve) 면적 계산
roc_auc = auc(fpr, tpr)

# ★ coolwarm 컬러맵에서 색상 추출 ★
cmap = plt.get_cmap('coolwarm')
color_roc = cmap(0.9)     # 따뜻한 붉은/오렌지색 톤 (Warm)
color_random = cmap(0.1)  # 차가운 짙은 푸른색 톤 (Cool)

# 3. ROC 커브 시각화
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color=color_roc, lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color=color_random, lw=2, linestyle='--', label='Random Guess')

# 그래프 꾸미기
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)

# 그래프 출력 및 저장 (필요시 주석 해제)
# plt.savefig('./Model/ROC_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. 확률값을 이진 클래스 예측값으로 변환 (임계값 0.5 기준)
# 이전 ROC 커브에서 구한 최적 임계값(best_thresh)을 0.5 대신 사용하셔도 좋습니다.
y_pred_class = (test_predictions > 0.5).astype(int)

# 2. 텍스트 형태의 종합 평가 지표 리포트 출력
print("================[ Classification Report ]=================")
# target_names는 실제 데이터의 0과 1 인코딩에 맞게 수정해주세요. (예: 0=Fail, 1=Pass)
print(classification_report(y_test, y_pred_class, target_names=['pass', 'fail']))
print("==========================================================")

# 3. 혼동 행렬 (Confusion Matrix) 시각화
cm = confusion_matrix(y_test, y_pred_class)

plt.figure(figsize=(6, 5))
# 앞서 사용하신 coolwarm 톤을 유지하여 통일성을 줍니다.
sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm', cbar=False,
            annot_kws={"size": 14, "weight": "bold"},
            xticklabels=['Predict: Pass', 'Predict: Fail'],
            yticklabels=['Actual: Pass', 'Actual: Fail'])

plt.title('Confusion Matrix', fontsize=15, pad=15)
plt.ylabel('True Label', fontsize=12, labelpad=10)
plt.xlabel('Predicted Label', fontsize=12, labelpad=10)

# plt.savefig('./Model/Confusion_Matrix.png', dpi=300, bbox_inches='tight') # 저장 필요시 주석 해제
plt.tight_layout()
plt.show()

## Notes

- Data source: 중소벤처기업부, Korea AI Manufacturing Platform(KAMP), 정밀가공 품질보증 AI 데이터셋, 스마트제조혁신추진단(㈜인터엑스), 2022.12.23., www.kamp-ai.kr
- This is a personal learning project based on the original KAMP analysis practice guide.
- The raw CSV and KAMP guidebook PDF are not included in this repository.
- Feature selection is performed on the training data to reduce test-set leakage.
